# 🚦 AmpelPilot - 3개 핵심 클래스 맞춤형 YOLOv8n 학습 & TFLite 변환

이 노트북은 유럽형 신호등 데이터셋(`ono-gedd7/pedestrian-traffic-light-puf4a`)에서
잡다한 라벨을 제거하고 **우리가 원하는 딱 3가지 핵심 클래스**만 추출하여 학습시킵니다:

1. 🔴 **`red`** : 보행자 빨간불 (정지 신호)
2. 🟢 **`green`** : 보행자 초록불 (보행 신호)
3. ⬛ **`pedestrian Traffic Light`** : 보행자 신호등 기둥/하우징 외형 (불 꺼진 신호등 감지용!)

---

### ⚡ 시작 전 확인
상단 메뉴 **[런타임] → [런타임 유형 변경] → 하드웨어 가속기: `T4 GPU`** 로 설정되어 있는지 확인하세요!

In [ ]:
# ============================================================
# [셀 1] Roboflow API 키 입력
# 무료 발급: https://app.roboflow.com 접속 ➔ Settings ➔ API Keys
# ============================================================
ROBOFLOW_API_KEY = "여기에_본인_API_키_입력"  # <--- 본인 키로 변경!
# ============================================================

In [ ]:
# [셀 2] 필수 라이브러리 설치 (10초)
!pip install -q ultralytics roboflow
print("✅ 패키지 설치 완료!")

In [ ]:
# [셀 3] 데이터셋 다운로드 & '3개 클래스(red, green, pedestrian Traffic Light)'만 자동 필터링
from roboflow import Roboflow
import os, glob, yaml

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("📥 유럽형 보행자 신호등 데이터셋 다운로드 중...")
project = rf.workspace("ono-gedd7").project("pedestrian-traffic-light-puf4a")
try:
    dataset = project.version(1).download("yolov8")
except Exception:
    dataset = project.version(4).download("yolov8")

print(f"✅ 원본 다운로드 완료: {dataset.location}")

# --- 3개 클래스만 남기기 위한 자동 필터링 작업 ---
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    orig_cfg = yaml.safe_load(f)

orig_names = orig_cfg.get('names', [])
if isinstance(orig_names, dict):
    orig_names = [orig_names[k] for k in sorted(orig_names.keys())]

print(f"\n🔎 원본 클래스 목록: {orig_names}")

# 매핑 규칙 정의
# 목표: 0: red, 1: green, 2: pedestrian Traffic Light
target_classes = ['red', 'green', 'pedestrian Traffic Light']
name_to_target_id = {}
for orig_id, name in enumerate(orig_names):
    name_clean = name.strip()
    if name_clean.lower() == 'red':
        name_to_target_id[orig_id] = 0
    elif name_clean.lower() == 'green':
        name_to_target_id[orig_id] = 1
    elif 'pedestrian' in name_clean.lower() or name_clean == 'pedestrian Traffic Light':
        name_to_target_id[orig_id] = 2

print(f"🎯 유지 및 재매핑 규칙: {name_to_target_id}")

# 모든 라벨 txt 파일 재가공 (불필요한 클래스 제거 & 3개 클래스 ID로 정렬)
total_kept = 0
for split in ['train', 'valid', 'test']:
    lbl_dir = os.path.join(dataset.location, split, 'labels')
    if not os.path.isdir(lbl_dir): continue
    for txt_file in glob.glob(os.path.join(lbl_dir, '*.txt')):
        with open(txt_file, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts: continue
            cls_id = int(parts[0])
            if cls_id in name_to_target_id:
                new_id = name_to_target_id[cls_id]
                new_lines.append(f"{new_id} {' '.join(parts[1:])}\n")
                total_kept += 1
        with open(txt_file, 'w') as f:
            f.writelines(new_lines)

# data.yaml 파일 업데이트
orig_cfg['names'] = target_classes
orig_cfg['nc'] = 3
with open(data_yaml_path, 'w') as f:
    yaml.safe_dump(orig_cfg, f)

print(f"\n✅ 필터링 완료! 총 {total_kept}개의 유효 라벨 유지됨")
print("📋 최종 확정 클래스:")
for i, n in enumerate(target_classes):
    print(f"   [{i}] {n}")

In [ ]:
# [셀 4] 3개 클래스 YOLOv8n 전이학습 시작 (약 10~15분)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=data_yaml_path,
    epochs=30,
    imgsz=640,
    batch=16,
    name='ampelpilot_3class',
    patience=8,
    save=True,
    plots=True
)

print("\n🎉 3개 클래스 전이학습 완료!")

In [ ]:
# [셀 5] 학습 결과 그래프 및 샘플 예측 확인
from IPython.display import Image, display

res_img = '/content/runs/detect/ampelpilot_3class/results.png'
if os.path.exists(res_img):
    print("📊 학습 결과 곡선:")
    display(Image(filename=res_img, width=800))

preds = glob.glob('/content/runs/detect/ampelpilot_3class/val_batch*_pred.jpg')
if preds:
    print("🔍 실제 신호등 감지 결과 샘플:")
    display(Image(filename=preds[0], width=600))

In [ ]:
# [셀 6] 안드로이드용 TFLite (Float16) 변환 (약 6MB)
best_weight = '/content/runs/detect/ampelpilot_3class/weights/best.pt'
model_best = YOLO(best_weight)

print("⚙️ TFLite 모델로 변환 중...")
exported = model_best.export(
    format='tflite',
    imgsz=640,
    half=True  # Float16 양자화
)

tflite_files = glob.glob('/content/runs/detect/ampelpilot_3class/weights/**/*.tflite', recursive=True)
if not tflite_files:
    tflite_files = glob.glob('/content/**/*.tflite', recursive=True)

for f in tflite_files:
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f"  📦 생성 완료: {f} ({size_mb:.2f} MB)")

In [ ]:
# [셀 7] labelmap.txt 생성 & 자동 다운로드
from google.colab import files

labelmap_path = '/content/labelmap.txt'
with open(labelmap_path, 'w') as f:
    for name in target_classes:
        f.write(name + '\n')

print(f"📄 최종 labelmap.txt 생성 완료:")
with open(labelmap_path, 'r') as f:
    print(f.read())

print("📥 파일 다운로드 시작...")
if tflite_files:
    files.download(tflite_files[0])
files.download(labelmap_path)

print("\n" + "="*60)
print("🎉 완벽합니다! 다운로드된 두 파일을 아래 위치에 넣으세요:")
print("   AmpelPilot/app/src/main/assets/")
print("   1) *.tflite 파일 ➔ detect.tflite 로 이름 변경")
print("   2) labelmap.txt ➔ 그대로 덮어쓰기")
print("="*60)